# Qwen3-TTS single-speaker fine-tune (single speaker)

Official recipe: <https://github.com/QwenLM/Qwen3-TTS/tree/main/finetuning>

**This is a FULL fine-tune, not LoRA** — upstream `sft_12hz.py` does
`AdamW(model.parameters())` and ships no PEFT path. (Community LoRA/QLoRA
adapters for this base do exist on the Hub, e.g. `aguken-ai/…-hi-LoRA-Finetuned-BNB-NF4`.)

**Runtime:** Runtime > Change runtime type > GPU. T4 works with the step-3 patches.

**Measured, not estimated.** The checkpoint is **914.6M params**, not 0.6B
(talker.model 754.8M + code_predictor 141.6M + speaker_encoder 8.9M +
text_projection 6.3M + codec_head 3.1M). Full fp32 AdamW = weights 3.7 + grads 3.7
+ states 7.3 = **14.6GB** against 14.56GB usable on a T4 — it OOMs inside
`optimizer.step()`. 8-bit Adam drops states to 1.8GB; observed peak ~13GB.

The 1.7B needs ~27GB and will OOM on anything below an A100.


In [ ]:
# 1. Check the GPU you actually got
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("torch", torch.__version__, "| bf16 supported:", torch.cuda.is_bf16_supported())

# --- SET THESE TO YOUR OWN ---------------------------------------------------
import os
DATASET_REPO = "your-username/your-voice-tts"   # private HF dataset repo, step 4
SPEAKER      = "myvoice"                        # name the voice registers under
os.environ["SPEAKER"] = SPEAKER                 # gen.py (step 7) reads it from env
# -----------------------------------------------------------------------------


In [ ]:
# 1b. Resource probe. Pure measurement -- it reads counters and changes nothing
#     about training, so numbers from a run with logging are directly comparable
#     to one without.
import subprocess, shutil, os
def res(tag=""):
    try:
        u, t = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total",
             "--format=csv,noheader,nounits"], text=True).strip().split("\n")[0].split(",")
        gpu = f"GPU {int(u)/1024:.1f}/{int(t)/1024:.1f}GB"
    except Exception:
        gpu = "GPU n/a"
    try:
        import psutil
        v = psutil.virtual_memory()
        ram = f"RAM {(v.total-v.available)/2**30:.1f}/{v.total/2**30:.1f}GB"
    except Exception:
        ram = "RAM n/a"
    d = shutil.disk_usage("/content")
    print(f"[res] {tag:24s} {gpu} | {ram} | DISK {d.used/2**30:.1f}/{d.total/2**30:.1f}GB")
res("baseline")


In [ ]:
# 2. Install + clone. Takes a few minutes.
!pip -q install -U qwen-tts accelerate
!git clone -q https://github.com/QwenLM/Qwen3-TTS.git
%cd /content/Qwen3-TTS/finetuning

In [ ]:
# 3. PATCH sft_12hz.py. Five things upstream assumes that are not true here.
# Reset first: an earlier run of this cell already edited the file, so the
# replaces below would silently miss their targets. Always patch from pristine.
!git -C /content/Qwen3-TTS checkout -- finetuning/sft_12hz.py
!pip -q install -U bitsandbytes

import torch, pathlib
p = pathlib.Path("sft_12hz.py"); s = p.read_text()

# a) flash_attention_2 needs Ampere+ (sm_80). sdpa ships with torch and works anywhere.
s = s.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')

# b) log_with="tensorboard" needs a logging_dir. init_trackers is never called and
#    nothing ever .log()s, so this is dead config -> drop it.
s = s.replace(', log_with="tensorboard"', '')

# c) T4 is Turing: no native bf16 (is_bf16_supported() says True only via emulation).
#    accelerator.prepare wraps the optimizer in a GradScaler, and fp16 weights + a
#    scaler raises "Attempting to unscale FP16 gradients" -- AMP needs fp32 masters.
#    So: fp32 weights, fp16 autocast.
if torch.cuda.get_device_capability()[0] < 8:
    s = s.replace('mixed_precision="bf16"', 'mixed_precision="fp16"')
    s = s.replace("torch_dtype=torch.bfloat16", "torch_dtype=torch.float32")

# d) UPSTREAM BUG. text_embedding is nn.Embedding(vocab, text_hidden_size=2048) in
#    both sizes, but talker hidden_size is 2048 on the 1.7B and 1024 on the 0.6B.
#    The model has a text_projection MLP (modeling_qwen3_tts.py:1575) that resizes
#    2048 -> hidden_size, and every inference path uses it; sft_12hz.py:89 does not.
#    On the 1.7B that is invisible (2048->2048); on the 0.6B it is a shape error.
#    Project first, then mask -- the MLP has bias=True, so masking last is required
#    to keep padded positions at zero.
old = "input_text_embedding = model.talker.model.text_embedding(input_text_ids) * text_embedding_mask"
new = ("input_text_embedding = model.talker.text_projection(\n"
       "                    model.talker.model.text_embedding(input_text_ids)) * text_embedding_mask")
assert old in s, "line 89 not found - upstream may have fixed this"
s = s.replace(old, new)

# e) OOM. This checkpoint is 914.6M params, not 0.6B (talker.model 754.8M +
#    code_predictor 141.6M + the rest). Full fp32 AdamW = weights 3.7 + grads 3.7
#    + states 7.3 = 14.6GB, against 14.56GB usable on a T4. Batch size is
#    irrelevant -- optimizer state does not depend on it. 8-bit Adam keeps the
#    full fine-tune and drops states 7.3GB -> 1.8GB.
if torch.cuda.get_device_properties(0).total_memory < 24e9:
    s = s.replace("from torch.optim import AdamW",
                  "from bitsandbytes.optim import AdamW8bit as AdamW")

p.write_text(s)
assert "flash_attention_2" not in s and "log_with" not in s
assert "talker.text_projection(" in s
print("optimizer:", "AdamW8bit" if "AdamW8bit" in s else "AdamW fp32")
print("sm_%d%d" % torch.cuda.get_device_capability(),
      "| mixed_precision:", "fp16" if 'mixed_precision="fp16"' in s else "bf16",
      "| weights:", "fp32" if "torch.float32" in s else "bf16")


In [ ]:
# 3b. sft_12hz.py does shutil.copytree(MODEL_PATH, ...) at the end of each epoch.
#     MODEL_PATH must therefore be a real local directory, not a hub id -- otherwise
#     it raises FileNotFoundError *after* the epoch has already been trained.
from huggingface_hub import snapshot_download
BASE = snapshot_download("Qwen/Qwen3-TTS-12Hz-0.6B-Base")
print(BASE)
!ls {BASE}
res("after base download")


In [ ]:
# 4. Pull the private dataset from HF instead of the browser upload widget.
from huggingface_hub import snapshot_download
import getpass, os
os.environ["HF_TOKEN"] = getpass.getpass("HF token (hf_...): ")
snapshot_download(DATASET_REPO, repo_type="dataset",
                  local_dir=".", token=os.environ["HF_TOKEN"])
!wc -l train_raw.jsonl && ls wavs | head -3
res("after dataset pull")


In [ ]:
# 5. Extract audio codes with the 12Hz tokenizer.
!python prepare_data.py \
  --device cuda:0 \
  --tokenizer_model_path Qwen/Qwen3-TTS-Tokenizer-12Hz \
  --input_jsonl train_raw.jsonl \
  --output_jsonl train_with_codes.jsonl
res("after tokenize")


In [ ]:
# 5b. Verify step 5 actually wrote codes before burning GPU time on step 6.
import json
rows = [json.loads(l) for l in open("train_with_codes.jsonl")]
assert len(rows) == 29, f"expected 29 rows, got {len(rows)}"
k = next(x for x in rows[0] if "code" in x.lower())
assert all(r[k] for r in rows), "some rows have empty codes"
print(len(rows), "rows | key:", k, "| first row code len:", len(rows[0][k]))


In [ ]:
!ls -la train_with_codes.jsonl && head -c 300 train_with_codes.jsonl

In [ ]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
# 6. Fine-tune. fp32 weights + fp16 AMP + 8-bit Adam ~= 9.1GB of params/grads/states.
#    lr 2e-5, 3 epochs: with under 2 minutes of audio this overfits fast.
#
#    The GPU sampler runs in the background because sft_12hz.py is a SUBPROCESS:
#    it releases all its memory on exit, so calling res() afterwards would just
#    report an idle card. Peak is only observable while it runs.
res("before training")
!nohup nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits -l 5 > gpu_train.log 2>&1 &

!python sft_12hz.py \
  --init_model_path {BASE} \
  --output_model_path output \
  --train_jsonl train_with_codes.jsonl \
  --batch_size 2 \
  --lr 2e-5 \
  --num_epochs 3 \
  --speaker_name {SPEAKER}

!pkill -f "query-gpu=memory.used,utilization.gpu" || true
import subprocess
rows = [r.split(",") for r in open("gpu_train.log").read().strip().splitlines() if "," in r]
if rows:
    peak = max(int(r[0]) for r in rows)
    print(f"[res] TRAINING PEAK GPU {peak/1024:.1f}GB over {len(rows)} samples "
          f"({len(rows)*5}s), max util {max(int(r[1]) for r in rows)}%")
res("after training")
!du -sh output/checkpoint-epoch-* 2>/dev/null


In [ ]:
%%writefile gen.py
# 7. Generation runs in a SUBPROCESS on purpose. A CUDA device-side assert tears
#    down the CUDA context for the whole process -- after one fires, every later
#    CUDA call in that same kernel fails, including a fresh from_pretrained on a
#    different checkpoint. In a subprocess the assert kills only the child and the
#    notebook kernel stays healthy, so no runtime restart is ever needed.
import argparse, os, time, torch, soundfile as sf
from qwen_tts import Qwen3TTSModel

TEXT = "I have been working on speech models for the last few weeks, mostly on my own laptop."

a = argparse.ArgumentParser()
a.add_argument("ckpt")
a.add_argument("--tag", default="")
a.add_argument("--greedy", action="store_true")
a.add_argument("--cpu", action="store_true")
a.add_argument("--speaker", default=os.environ.get("SPEAKER", "myvoice"))
a.add_argument("--fp32", action="store_true")
a = a.parse_args()

if a.cpu:
    dev, dt = "cpu", torch.float32
else:
    # T4 is Turing: is_bf16_supported() says True but only via emulation, which crawls.
    dev = "cuda:0"
    if a.fp32:
        dt = torch.float32   # fp16 generation failed on T4; fp32 inference is only 3.7GB
    else:
        dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("device", dev, "| dtype", dt, "| greedy", a.greedy)

t0 = time.time()
tts = Qwen3TTSModel.from_pretrained(a.ckpt, device_map=dev, dtype=dt,
                                    attn_implementation="sdpa")
print("  loaded in", round(time.time() - t0), "s")
print("  speakers:", tts.model.get_supported_speakers())

kw = {"do_sample": False} if a.greedy else {}
t1 = time.time()
wavs, sr = tts.generate_custom_voice(text=TEXT, language="English",
                                     speaker=a.speaker, max_new_tokens=1024, **kw)
out = "ft_" + a.ckpt.rstrip("/").split("-")[-1] + a.tag + ".wav"
sf.write(out, wavs[0], sr)
print("  wrote", out, round(len(wavs[0]) / sr, 2), "s audio in",
      round(time.time() - t1), "s")
peak = torch.cuda.max_memory_allocated() / 2**30 if dev != "cpu" else 0
import resource
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20   # macOS bytes / linux KB
print(f"  peak GPU {peak:.2f}GB | peak RSS {rss:.2f}GB")


In [ ]:
# 7a. Every checkpoint x {fp16, fp32} x {sampled, greedy}.
#     BOTH streams are printed. The previous version printed stdout OR stderr, so
#     every traceback was discarded -- all six runs "failed silently".
import subprocess, glob
for ck in sorted(glob.glob("output/checkpoint-epoch-*")):
    for tag, extra in [("", []), ("_greedy", ["--greedy"]),
                       ("_fp32", ["--fp32"]), ("_fp32_greedy", ["--fp32", "--greedy"])]:
        print("==", ck, tag or "(fp16 sampled)")
        r = subprocess.run(["python", "gen.py", ck, "--tag", tag] + extra,
                           capture_output=True, text=True)
        print(r.stdout.strip())
        if r.returncode != 0:
            print("  RC", r.returncode, "STDERR:", r.stderr.strip()[-800:])


In [ ]:
# 7b. Exact diagnosis, only if EVERY run above failed. On CPU, PyTorch raises a
#     real IndexError naming the offending index instead of an async device
#     assert. Hypothesis: the talker samples a codec_0 token >= 2048, which then
#     indexes code_predictor's embeddings -- talker vocab_size is 3072 but
#     code_predictor_config.vocab_size is 2048, and the 2048-3071 band holds
#     speaker/special ids (the fine-tuned speaker = 3000). Slow but definitive.
!python gen.py output/checkpoint-epoch-2 --tag _cpu --cpu


In [ ]:
# 7c. Listen in the browser before scoring anything.
import glob
from IPython.display import Audio, display
for f in sorted(glob.glob("ft_*.wav")):
    print(f); display(Audio(f))


In [ ]:
# 8. Push the samples back to HF - files.download() is the same browser-session
#    widget that failed in step 4, so it will not work from a terminal client.
#    Step 4 asked for a READ token; uploading needs WRITE, so ask again here
#    rather than keeping a write token live for the whole session.
import glob, getpass
from huggingface_hub import HfApi
api = HfApi(token=getpass.getpass("HF WRITE token (hf_...): "))
for f in sorted(glob.glob("ft_*.wav")):
    api.upload_file(path_or_fileobj=f, path_in_repo="finetuned/" + f,
                    repo_id=DATASET_REPO, repo_type="dataset")
    print("pushed", f)
# Then on the Mac:
#   hf download $DATASET_REPO --repo-type dataset \
#       --include "finetuned/*" --local-dir tts_models/voice_clone/dataset/finetune_out


## Result so far

Scored on the same calibrated timbre scale as zero-shot
(ceiling 0.9956 = you vs you, floor 0.8432 = mean of Kokoro/Piper/say/Inflect):

| | cosine | scale |
|---|---|---|
| zero-shot ICL | 0.9272 | **55.1%** |
| Kokoro (a floor voice) | 0.8626 | 12.7% |
| **fine-tuned, epoch 2** | **0.8577** | **9.5%** |

The fine-tune scored **below a generic TTS voice**. 29 utterances / 1.7 minutes
is not enough: the smallest run reported on the Hub for this base is 200 samples
(`duarteocarmo`, `zero0303`), and those use lr 1e-6 to 5e-6 — ours was 2e-5.

Next run: more audio first, then lr 5e-6 and more epochs. Not more epochs alone.
